To use this script, label the grains interiors with magenta in image j (or you can change the color threshold to something other than magenta, imagej will tell you the value), and the boundaries with white. Anything unlabelled gets mapped to background

There are two remapping modes, selectable via `mapping_mode` in the cell below:

- `"three_class"`: background (0) / interior (1) / boundary (2). Interior is taken from `grain_interior` (magenta), boundary from `grain_bound` (white, ≥ `BOUND_MIN`), everything else is background. This is the original behaviour.
- `"boundary_interior"`: only two classes. White pixels (≥ `BOUND_MIN`) become boundary (2), and **everything else** becomes interior (1). There is no background class in this mode.

**Note:** the "white" boundary value drifts between exports — older annotations saved it as 252, newer ones (030/040) as 253 — so boundaries are matched with a `>= BOUND_MIN` (250) threshold rather than an exact value. An exact match against the wrong value produces a mask with *zero* boundary pixels (one giant interior blob); the sanity check at the bottom guards against that.

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

name="annotations/annotated_cropped_Wilson180_0.25__surface1_05.tif"
#name="/Users/Alexander/Repositories/segmenteverygrain/annotations/annotated_cropped_prac7_etched_020.tif"

# Select the remapping mode:
#   "three_class"       -> background (0) / interior (1, from grain_interior) / boundary (2, white)
#   "boundary_interior" -> boundary (2) where pixel >= BOUND_MIN, interior (1) everywhere else (no background)
mapping_mode = "three_class"

# White boundary lines come out at slightly different values depending on the export
# (252 in the older annotations, 253 in 030/040), so match with a threshold, not equality.
BOUND_MIN = 250
grain_interior=105

# Load grayscale image
img = cv2.imread(name, cv2.IMREAD_GRAYSCALE)
print(img.dtype, img.min(), img.max())
print(np.unique(img))

unique, counts = np.unique(img, return_counts=True)

# Combine and sort by count descending
value_counts = sorted(zip(unique, counts), key=lambda x: x[1], reverse=True)

print("Most frequent pixel values:")
for val, cnt in value_counts[:10]:
    print(f"Value {val}: {cnt} pixels")

plt.bar([v for v, c in value_counts], [c for v, c in value_counts]) #plot values, for debugging color issue
plt.xlabel("Pixel value")
plt.ylabel("Count")
plt.title("Pixel value frequencies")
plt.show()

# Apply thresholding / labeling
if mapping_mode == "three_class":
    # Everything starts as background (0)
    labels = np.zeros_like(img, dtype=np.uint8)
    labels[img == grain_interior] = 1
    labels[img >= BOUND_MIN] = 2
elif mapping_mode == "boundary_interior":
    # Everything starts as interior (1), white becomes boundary (2)
    labels = np.ones_like(img, dtype=np.uint8)
    labels[img >= BOUND_MIN] = 2
else:
    raise ValueError(f"Unknown mapping_mode: {mapping_mode!r}")

# Sanity check: an annotation whose boundary value didn't match would produce ~no boundary
# pixels and the whole image collapses into one interior blob.
bound_frac = float((labels == 2).mean())
assert bound_frac > 0.01, (
    f"Only {bound_frac:.2%} boundary pixels found -- boundary value probably not >= {BOUND_MIN}; "
    "check the pixel-value histogram above."
)

# Build output filename: <originalname>_mask.tif
directory, filename = os.path.split(name)
base, ext = os.path.splitext(filename)

output_filename = f"{base}_mask{ext}"
output_path = os.path.join(directory, output_filename)

# Save result
cv2.imwrite(output_path, labels)


# Grayscale visualization (1–2 mapped properly)
plt.figure(figsize=(6, 6))
plt.imshow(labels, cmap="gray", vmin=0, vmax=2)
plt.colorbar(label="Label value")
if mapping_mode == "three_class":
    plt.title("Mask visualization (0=background, 1=grain, 2=boundary)")
else:
    plt.title("Mask visualization (1=grain interior, 2=boundary)")
plt.axis("off")
plt.show()

num_grain_pixels = np.sum(labels == 1)
num_boundary_pixels = np.sum(labels == 2)

print(f"Grain interior pixels (1): {num_grain_pixels}")
print(f"Grain boundary pixels (2): {num_boundary_pixels}")

print(f"Mask remapped (mode: {mapping_mode})")
